# BEVPlace++ on KITTI: experiments

Runtime -> Change runtime type -> **T4 GPU** (or better). Set cell 1, then **Runtime -> Run all**.
`MODE` in cell 1 says what this run is for (train / eval / smoke); every cell does only what that mode needs.

- **BEV variant**: built on the Azure VM (`METHOD=bev bash scripts/azure_build.sh`), saved in `MyDrive/lpr/bev/<variant>/`.
- **Run**: `MyDrive/lpr/runs/bev/<run>/` (`config.json`, `metrics.csv`, a checkpoint per epoch).
- **Official checkpoint** (`EVAL_OFFICIAL`): the authors' network evaluated on our BEV images, in `MyDrive/lpr/runs/bev/official_<variant>/`.
- **Evaluations** (`EVALS`, types in `src/lpr/evals.py`): `<run>/eval/<type>/<checkpoint>/`; tables in
  `MyDrive/lpr/runs_summary.csv` and `MyDrive/lpr/evals_summary.csv`.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/dov85/lidar-place-recognition"
DRIVE = "/content/drive/MyDrive/lpr"
SEQ = "00"                          # training sequence
# What this run of the notebook does. "Runtime -> Run all" works in every mode:
#   "train": data, training (a new run, or the continuation of an unfinished one; a finished run
#            is not trained again), then the evaluations
#   "eval":  only the evaluations of an existing run (nothing is trained or built)
#   "smoke": a short training test (time and GPU memory), nothing is saved to Drive
MODE = "train"
EVAL_CKPT = "last"                  # "last" or a checkpoint file of the run, e.g. "epoch_18.pth"
EVALS = ['seqs_heading', 'compass']   # types in src/lpr/evals.py (EVAL_TYPES)
EVAL_OFFICIAL = True                # also evaluate the authors' checkpoint (once; then it is skipped)
CHECK_BEV = False                   # compare our BEV images with the official script on 3 scans

# BEV variant: only what differs from the defaults (src/lpr/experiments.py, BEV_DEFAULTS):
#   half_size=40.0, z_abs=40.0, voxel=0.4, res=0.4, max_count=10, ground="none"
BEV_TAG = ""          # name for a changed variant; empty = date + hash
BEV_PARAMS = dict()                 # e.g. dict(ground="patchwork")

# Training run: only what differs from the defaults (BEV_TRAIN_DEFAULTS, the official values):
#   train_frames=3000, pos_thr=5.0, neg_thr=7.0, epochs=40, lr=1e-4, batch=4, negatives=10,
#   margin=0.3, rot_aug=True, seed=1024,
#   arch="rein"   ("plain": the same CNN + NetVLAD in one pass, without REIN's 8 rotations, i.e. no
#                  built-in rotation invariance; with rot_aug=True/False for with/without rotated images)
RUN_TAG = "without_Rein"          # name for a changed run; empty = date + hash
TRAIN_PARAMS = dict(epochs=20,rot_aug=False,arch="plain")               # e.g. dict(epochs=20)

KITTI = "/content/kitti"            # poses (and a few raw scans for CHECK_BEV)

## 2. Drive, GPU, code, poses, Drive layout

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, json, subprocess, importlib, glob
assert MODE in ("train", "eval", "smoke"), MODE
os.makedirs(DRIVE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

url = REPO_URL
try:  # private repo: store a token in Colab Secrets as GITHUB_TOKEN
    url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass
if not os.path.exists("/content/lpr"):
    subprocess.run(["git", "clone", "-q", url, "/content/lpr"], check=True)
else:
    subprocess.run(["git", "-C", "/content/lpr", "pull", "-q"], check=True)
%cd /content/lpr
!git log --oneline -1
!bash scripts/setup_third_party.sh
!pip install -q -r requirements.txt faiss-cpu

sys.path.insert(0, "/content/lpr/src")
import lpr.experiments as ex, lpr.evals as ev
importlib.reload(ex); importlib.reload(ev)
missing_poses = [s for s in ev.ALL_SEQS if not os.path.exists(f"{KITTI}/poses/{s}.txt")]
if missing_poses:
    !python scripts/download_kitti.py --out {KITTI} --seqs {" ".join(missing_poses)} --poses_only

# older results into the current Drive layout (does nothing once done)
if os.path.exists(f"{DRIVE}/runs/ndt_kitti00"):
    !python scripts/migrate_stage1.py --drive {DRIVE} --apply
!python scripts/migrate_layout.py --drive {DRIVE} --apply

## 3. Names of the BEV variant and the run

In [ ]:
VARIANT, BEV_FULL, bev_exists = ex.resolve_variant(DRIVE, "bev", BEV_PARAMS, BEV_TAG)
VDIR = f"{DRIVE}/bev/{VARIANT}"         # permanent copy on Drive
BEV = f"/content/bev/{VARIANT}"         # working copy on this machine

RUN, TRAIN_FULL, run_state = ex.resolve_run(DRIVE, dict(TRAIN_PARAMS, seq=SEQ), VARIANT, RUN_TAG, method="bev")
RUN_DIR = f"{DRIVE}/runs/bev/{RUN}"
print("mode:       ", MODE)
print("BEV variant:", VARIANT, "(on Drive)" if bev_exists else "(new: build it on Azure, see below)")
print(json.dumps(BEV_FULL))
print("run:        ", RUN, f"({run_state})")
print(json.dumps(TRAIN_FULL))
if MODE == "eval" and run_state == "new":
    print("MODE = 'eval' and no run with these parameters: only the official checkpoint is evaluated")
defaults = ex.BEV_DEFAULTS
args = " ".join(f"--{k} {v}" for k, v in BEV_FULL.items() if v != defaults[k])
print("\nTo build the variant on the Azure VM (inside tmux, in the repository folder):")
print(f'  git pull && METHOD=bev VARIANT={VARIANT}' + (f' ARGS="{args}"' if args else "") +
      ' bash scripts/azure_build.sh "<SAS URL>"')

t = TRAIN_FULL
train_args = (f"--bev_dir {BEV} --kitti_root {KITTI} --bev_variant {VARIANT} --seq {t['seq']} "
              f"--train_frames {t['train_frames']} --pos_thr {t['pos_thr']} --neg_thr {t['neg_thr']} "
              f"--epochs {t['epochs']} --lr {t['lr']} --batch {t['batch']} --negatives {t['negatives']} "
              f"--margin {t['margin']} --seed {t['seed']}" + ("" if t["rot_aug"] else " --no_rot_aug")
              + (f" --arch {t['arch']}" if t["arch"] != "rein" else ""))

## 4. BEV images: from Azure to Drive to this machine
Copies the tars of the variant from the blob container (SAS URL in Colab Secrets as `AZURE_SAS`, permissions read + list)
into `VDIR`, then unpacks them into `BEV`. Training needs sequence 00, the evaluations every sequence.

In [ ]:
need = [SEQ] if MODE == "smoke" else ev.ALL_SEQS
tar_of = lambda s: f"{VDIR}/{ex.archive_name('bev', s)}"
missing = [s for s in need if not os.path.exists(tar_of(s))]
if missing:
    res = subprocess.run(["python", "scripts/fetch_from_azure.py", "--sas_url", userdata.get("AZURE_SAS"), "--method", "bev",
                          "--variant", VARIANT, "--seqs", *missing, "--drive", DRIVE],
                         capture_output=True, text=True, cwd="/content/lpr")
    print(res.stdout, res.stderr[-2000:])
for s in need:
    if os.path.exists(tar_of(s)) and not os.path.exists(f"{BEV}/{s}/000000.png"):
        os.makedirs(BEV, exist_ok=True)
        subprocess.run(["tar", "-xf", tar_of(s), "-C", BEV], check=True)
HAVE = [s for s in need if os.path.exists(f"{BEV}/{s}/000000.png")]
print("BEV images on this machine for:", {s: len(os.listdir(f"{BEV}/{s}")) for s in HAVE})
if MODE in ("train", "smoke") and SEQ not in HAVE:
    raise SystemExit(f"sequence {SEQ} missing: build the variant on Azure first (command printed in cell 3)")

### 4b. Optional (`CHECK_BEV = True`): our BEV images = the official script's
Downloads 3 scans of sequence 00 and compares `lpr.bev` with `third_party/BEVPlace2/datasets/gen_bev_images.py`
pixel by pixel. Expected last line: `IDENTICAL`.

In [ ]:
if CHECK_BEV:
    !pip install -q open3d
    !python scripts/download_kitti.py --out /content/kitti_check --seqs 00 --first 3
    scans = " ".join(f"/content/kitti_check/sequences/00/velodyne/{i:06d}.bin" for i in range(3))
    !python tests/test_bev_matches_official.py --scans {scans}
else:
    print("skipped (CHECK_BEV = False)")

## 5. Smoke test (MODE = "smoke"): 20 iterations
Seconds per iteration x 750 is roughly one epoch, plus the descriptor cache at the start of each epoch.
If 4 x 12 images do not fit in GPU memory, training switches to one tuple per forward pass (`split_forward`).

In [ ]:
if MODE == "smoke":
    !rm -rf /content/smoke
    !python scripts/train_bev_kitti.py {train_args} --log_dir /content/smoke --epochs 1 --max_iters 20 --eval_every 0 2>&1 | grep -v Warning
else:
    print("skipped (MODE =", MODE + ")")

## 6. Training (MODE = "train"; continues an unfinished run, also inside an epoch)

In [ ]:
if MODE == "train":
    print("run:", RUN_DIR, f"({run_state})")
    !python scripts/train_bev_kitti.py {train_args} --log_dir {RUN_DIR} 2>&1 | grep -v Warning
else:
    print("skipped (MODE =", MODE + ")")

## 7. Evaluations (`EVALS`, checkpoint `EVAL_CKPT`; the official checkpoint with `EVAL_OFFICIAL`)
Each type is stored in `<run>/eval/<type>/<checkpoint>/` and is not repeated once complete.

In [ ]:
import pandas as pd
from IPython.display import Image, display

def run_evals(method, run_dir, ckpt, data_dir):
    """Every type in EVALS on one checkpoint; results in <run_dir>/eval/<type>/<checkpoint>/."""
    for e in EVALS:
        if method not in ev.EVAL_TYPES[e]["methods"]:
            continue
        res = subprocess.run(["python", "scripts/evaluate.py", "--method", method, "--run_dir", run_dir,
                              "--ckpt", ckpt, "--eval_type", e, "--data_dir", data_dir, "--kitti_root", KITTI],
                             capture_output=True, text=True, cwd="/content/lpr")
        print(res.stdout if res.returncode == 0 else res.stdout[-2000:] + res.stderr[-3000:])
        if res.returncode == 0:
            out = ev.eval_dir(run_dir, e, ev.resolve_ckpt(run_dir, ckpt))
            for png in sorted(glob.glob(f"{out}/*.png")):
                display(Image(png))

OFFICIAL_CKPT = "/content/lpr/third_party/BEVPlace2/runs/Aug08_10-17-29/model_best.pth.tar"
OFF_DIR = f"{DRIVE}/runs/bev/official_{VARIANT}"
if MODE == "smoke":
    print("skipped (MODE = smoke)")
else:
    if glob.glob(f"{RUN_DIR}/epoch_*.pth"):
        run_evals("bev", RUN_DIR, EVAL_CKPT, BEV)
    else:
        print("no checkpoint in", RUN_DIR)
    if EVAL_OFFICIAL:
        if not os.path.exists(f"{OFF_DIR}/config.json"):
            os.makedirs(OFF_DIR, exist_ok=True)
            ex.write_json(f"{OFF_DIR}/config.json", {"run": os.path.basename(OFF_DIR), "method": "bev",
                          "bev_variant": VARIANT, "status": "finished",
                          "params": {"source": "official checkpoint Aug08_10-17-29/model_best.pth.tar"}})
        run_evals("bev", OFF_DIR, OFFICIAL_CKPT, BEV)

## 8. Tables of all runs and evaluations, training curves

In [ ]:
import matplotlib.pyplot as plt
!python scripts/summarize_runs.py --drive {DRIVE} > /dev/null
display(pd.read_csv(f"{DRIVE}/runs_summary.csv", dtype={"run": str}))
if os.path.exists(f"{RUN_DIR}/metrics.csv"):
    m = pd.read_csv(f"{RUN_DIR}/metrics.csv").drop_duplicates("epoch", keep="last")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(m.epoch, m.train_loss, marker="o"); ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
    if "R@1" in m:
        ax[1].plot(m.epoch, m["R@1"], marker="o", label="R@1"); ax[1].plot(m.epoch, m["R@1%"], marker="o", label="R@1%")
    ax[1].set_title(f"KITTI {SEQ} recall (%), monitoring only  {RUN}"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.show()

## 9. NDT-Transformer vs BEVPlace++ (every run with a `seqs_heading` evaluation)

In [ ]:
p = f"{DRIVE}/evals_summary.csv"
if os.path.exists(p):
    e = pd.read_csv(p, dtype={"seq": str, "run": str, "checkpoint": str})
    sh = e[e.eval_type == "seqs_heading"]
    print("Recall@1 per sequence")
    display(sh.pivot_table(index=["method", "run", "checkpoint"], columns="seq", values="R@1"))
    cols = [c for c in sh.columns if c.startswith("R@1 ")]
    for s in ("02", "08"):
        print(f"KITTI {s}: Recall@1 per heading group (share of queries in the % columns)")
        g = sh[sh.seq == s].set_index(["method", "run", "checkpoint"])
        display(g[[c for c in g.columns if c.startswith("% ")] + cols])
    cp = e[e.eval_type == "compass"]
    if len(cp):
        print("Compass: Recall@1 per sequence (columns) and compass error sigma in degrees (rows)")
        display(cp.pivot_table(index=["method", "run", "checkpoint", "sigma_deg"], columns="seq", values="R@1"))
    tn = e[e.eval_type == "tnet"]
    if len(tn):
        print("T-Net check (scans rotated by 'angle')")
        display(tn.dropna(axis=1, how="all").set_index(["run", "checkpoint", "angle"])
                .drop(columns=["method", "eval_type", "complete", "seq"], errors="ignore"))